# run on scc

Copy of `sites_2.ipynb` that builds the short/long site CSVs from `01_selected_sites_raw2.csv`
instead of `01_selected_sites_raw.csv`. Outputs `02_selected_sites_short_2.csv` and
`03_selected_sites_long_2.csv`.

In [1]:
import pandas as pd
import json
from pathlib import Path
import ipywidgets as widgets
from ipyleaflet import AwesomeIcon, Map, Marker, Polygon, ScaleControl, basemaps, basemap_to_tiles, TileLayer

In [2]:
DATA_DIR = Path.cwd() / 'data'
print(f"{DATA_DIR=}")
INFO_DIR = Path.cwd() / 'info'
print(f"{INFO_DIR=}")
GEOJSON_PATH = Path('/projectnb/planet/PLSP/geojson')
print(f'{GEOJSON_PATH=}')

DATA_DIR=PosixPath('/projectnb/modislc/users/fache/src/PLSP/code/selected_sites_info/data')
INFO_DIR=PosixPath('/projectnb/modislc/users/fache/src/PLSP/code/selected_sites_info/info')
GEOJSON_PATH=PosixPath('/projectnb/planet/PLSP/geojson')


# Create Selected Sites Short/Long Format CSVs

In [3]:
# from https://phenocam.nau.edu/webcam/network/table/
phenocam_df = pd.read_csv(DATA_DIR / 'phenocam_site_table.csv')

In [4]:
# from https://ameriflux.lbl.gov/sites/site-search/ export results
flux_df = pd.read_csv(DATA_DIR / 'AmeriFlux-site-search-results-202607161927.tsv', sep='\t')
flux_df.rename(columns={c: f'(f){c}' for c in list(flux_df.columns)}, inplace=True)
flux_df.rename(columns={'(f)Latitude (degrees)': '(f)latitude', '(f)Longitude (degrees)': '(f)longitude'}, inplace=True)

In [5]:
"""
from manual

01_selected_sites_raw2.csv uses different column names than 01_selected_sites_raw.csv,
so rename to the schema the rest of the notebook expects:
  plsp_product_id   -> site_full
  plsp_raw_id       -> site_name       (this is the geojson file name)
  ameriflux_id      -> site_id
  site_name         -> site_name_long  (human readable name)
  plsp_site_number  -> plsp_number
"""

sites_of_interest_df = pd.read_csv(INFO_DIR / '01_selected_sites_raw2.csv')
sites_of_interest_df.rename(
    columns={
        'plsp_product_id': 'site_full',
        'site_name': 'site_name_long',
        'plsp_raw_id': 'site_name',
        'ameriflux_id': 'site_id',
        'plsp_site_number': 'plsp_number',
    },
    inplace=True,
)
sites_of_interest_df

,site_full,site_name,site_id,neon_id,site_name_long,site_name_2,phenocam1,phenocam2,is_neon,neon_domain,is_focus,plsp_number
0,US-xSR_NEON_Santa_Rita_Experimental_Range,Santa_Rita_Experimental_Range_NEON,US-xSR,SRER,Santa Rita Experimental Range NEON,NaN,NEON.D14.SRER.DP1.00042,NEON.D14.SRER.DP1.00033,1,D14,1,NaN
1,US-Wkg_Walnut_Gulch_Kendall_Grasslands,Walnut_Gulch_Kendall_Grasslands,US-Wkg,NaN,Walnut Gulch Kendall Grasslands,NaN,kendall,NaN,0,D14,1,103.0


In [6]:
"""
merge dfs

raw2 has two phenocams per site, so the phenocam table is merged twice,
with (p1)/(p2) prefixes.
"""

data_all_df = sites_of_interest_df.copy()

for n in (1, 2):
    cam_df = phenocam_df.rename(columns={c: f'(p{n}){c}' for c in list(phenocam_df.columns)})
    cam_df = cam_df.rename(columns={f'(p{n})site_name': f'phenocam{n}'})
    data_all_df = pd.merge(data_all_df, cam_df, on=f'phenocam{n}', how='left')

data_all_df = pd.merge(
    data_all_df.assign(temp_key=data_all_df['site_id'].str.upper()),
    flux_df.assign(temp_key=flux_df['(f)Site ID'].str.upper()),
    on='temp_key',
    how='left'
).drop(columns=['temp_key', '(f)Site ID'])

In [7]:
"""
load geometry coordinates
"""

def read_geometry(row):
    # for ~2 sites, the geojson name is different than site_name, use site_name_2
    try:
        with open(GEOJSON_PATH / f'{row["site_name"]}.geojson', "r") as f:
            geo = json.load(f)
            geo = geo['features'][0]['geometry']['coordinates']
    except FileNotFoundError:
        with open(GEOJSON_PATH / f'{row["site_name_2"]}.geojson', "r") as f:
            geo = json.load(f)
            geo = geo['features'][0]['geometry']['coordinates'][0]
    
    # some features have extra nesting layer
    if len(geo) == 1:
        geo = geo[0]
    
    # flip from (lon, lat) to (lat, lon)
    geo = [[p[1], p[0]] for p in geo]

    return geo
        

data_all_df['coordinates'] = data_all_df.apply(read_geometry, axis=1)

In [8]:
"""
dtype conversion
"""

data_all_df['plsp_number'] = data_all_df['plsp_number'].astype('Int64')

In [9]:
short_info_columns = [
    'is_focus', 'site_full', 'site_id', 'site_name', 'site_name_2', 'site_name_long', 'plsp_number',
    'is_neon', 'neon_id', 'neon_domain', '(f)latitude', '(f)longitude',
    'phenocam1', '(p1)latitude', '(p1)longitude', 'phenocam2', '(p2)latitude', '(p2)longitude',
    'coordinates'
]

long_info_columns = [
    *short_info_columns,
    '(f)AmeriFlux FLUXNET DOI', '(f)Site Start', '(f)Site End',
    '(p1)site_url', '(p1)date_first', '(p1)date_last',
    '(p2)site_url', '(p2)date_first', '(p2)date_last'
]

data_all_df[short_info_columns].to_csv(INFO_DIR / '02_selected_sites_short_2.csv', index=False)
data_all_df[long_info_columns].to_csv(INFO_DIR / '03_selected_sites_long_2.csv', index=False)

# Create Interactive Map of Sites

In [10]:
def create_map(df):
    # center map on mean coordinates
    center_lat = df["(f)latitude"].mean()
    center_lon = df["(f)longitude"].mean()
    m = Map(center=[center_lat, center_lon], zoom=5, basemap=basemap_to_tiles(basemaps.Esri.WorldImagery))
    boundary_layer = TileLayer(url="https://server.arcgisonline.com/ArcGIS/rest/services/Reference/World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}", attribution="Tiles © Esri — Source: Esri, DeLorme, HERE")
    m.add(boundary_layer)
    m.add(ScaleControl(position='bottomleft'))

    blue_icon = AwesomeIcon(name="info-circle", marker_color="blue", icon_color="white")
    red_icon = AwesomeIcon(name="info-circle", marker_color="red", icon_color="white")

    for _, row in df.iterrows():
        # ---------- site marker
        marker1_coords = (row["(f)latitude"], row["(f)longitude"])
        hover_text_1 = (
            f"Site Name: {row['site_name']}\n"
            f"Site ID: {row['site_id']}\n"
            f"Coordinates: {marker1_coords}"
        )
        marker1 = Marker(
            location=marker1_coords,
            icon=red_icon if row['is_focus'] else blue_icon,
            draggable=False,
            title=hover_text_1,
        )
        popup_html_1 = widgets.HTML(
            value=f"<b>Site Name:</b> {row['site_name']}<br>"
            f"<b>Site ID:</b> {row['site_id']}<br>"
            f"<b>Coordinates:</b> {marker1_coords}"
        )
        marker1.popup = popup_html_1
        m.add(marker1)

        # ---------- site polygon
        blue_polygon = Polygon(
            locations=row['coordinates'],
            color="blue",
            fill_color="blue",
            fill_opacity=0.15,
            weight=2,
        )
        m.add(blue_polygon)

    return m

In [11]:
m = create_map(data_all_df)
display(m)
m.save('info/02_selected_sites_short_2.html', title='Selected Sites')

Map(center=[np.float64(31.8236), np.float64(-110.3887)], controls=(ZoomControl(options=['position', 'zoom_in_t…